In [26]:
import joblib
import pandas as pd

from predict_flows import get_expected_features, prepare_inference_frame, resolve_model_path


In [27]:
model_path = resolve_model_path("model.pkl")
model = joblib.load(model_path)

traffic_df = pd.read_csv("traffic2.pcap_Flow.csv")
print(f"Loaded rows: {len(traffic_df)}")
traffic_df.head()


model.pkl not found; using safeforge_model.pkl instead.
Loaded rows: 146


,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,140.82.113.21-10.157.65.13-443-53698-6,10.157.65.13,53698,140.82.113.21,443,6,31/03/2026 03:05:37 PM,1174499,18,19,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No Label
1,140.82.113.21-10.157.65.13-443-53698-6,10.157.65.13,53698,140.82.113.21,443,6,31/03/2026 03:05:38 PM,29054,2,1,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No Label
2,10.157.65.154-10.157.65.13-53-60139-6,10.157.65.13,60139,10.157.65.154,53,6,31/03/2026 03:05:39 PM,183510,5,6,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No Label
3,10.157.65.154-10.157.65.13-53-50221-6,10.157.65.13,50221,10.157.65.154,53,6,31/03/2026 03:05:39 PM,183875,5,6,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No Label
4,10.157.65.154-10.157.65.13-53-62649-6,10.157.65.13,62649,10.157.65.154,53,6,31/03/2026 03:05:40 PM,143415,5,6,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No Label


In [28]:
expected_features = get_expected_features(model)
all_features, missing_columns, derived_columns = prepare_inference_frame(traffic_df, expected_features)

if derived_columns:
    print("Derived columns:", derived_columns)

if missing_columns:
    print("Missing columns filled with 0:", missing_columns)
else:
    print("No training-feature columns are missing.")

all_features.head()


Using schema from model.feature_names_in_ (78 features).
Derived columns: ['Fwd Header Length.1 <- Fwd Header Length']
No training-feature columns are missing.


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
0,443,1174499,18,19,16312.0,6730.0,1300.0,0.0,906.222222,584.255037,...,14,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,443,29054,2,1,24.0,0.0,24.0,0.0,12.000000,16.970563,...,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,53,183510,5,6,31.0,196.0,29.0,0.0,6.200000,12.774976,...,2,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,53,183875,5,6,31.0,220.0,29.0,0.0,6.200000,12.774976,...,2,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,53,143415,5,6,47.0,240.0,45.0,0.0,9.400000,19.919839,...,2,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [29]:
predictions = model.predict(all_features)

output_df = traffic_df.copy()
output_df["Predicted Label"] = predictions
output_df.to_csv("predictions.csv", index=False)

print("Saved predictions.csv")
print(output_df["Predicted Label"].value_counts())

output_df[["Predicted Label"]]


Saved predictions.csv
Predicted Label
BENIGN    146
Name: count, dtype: int64


,Predicted Label
0,BENIGN
1,BENIGN
2,BENIGN
3,BENIGN
4,BENIGN
...,...
141,BENIGN
142,BENIGN
143,BENIGN
144,BENIGN
